# 03 — Cleaning & Feature Engineering (Phase 2)

## Scope: exploratory expansion covering ~80% of FEMA PA declarations

Phase 2 is an **exploratory expansion** of Phase 1, not a replacement. It demonstrates that
adding event-level physical intensity data improves funding tier prediction for the event types
where such data exists. Phase 1 remains the universal model for all disaster types.

**NOAA Storm Events covers hydrometeorological events only:**

| Covered by Phase 2 | Not covered (Phase 2 scope) |
|---|---|
| Hurricane, Flood, Severe Storm | Earthquake → needs USGS ShakeMap |
| Winter Storm, Tornado | Biological → needs CDC surveillance |
| Wildfire, Coastal Flood | Chemical/Human Cause → too rare to model |

Hydrometeorological events represent ~80% of FEMA PA declarations by count.
Non-hydrometeorological event types are excluded from this notebook to prevent
zero-inflation of NOAA features — including earthquake/biological rows would mean
those records have NOAA intensity = 0 not because the event was mild, but because
NOAA has no coverage for those event types, which would confound the model.

## New NOAA features (for covered event types)
- `noaa_peak_magnitude`: maximum wind speed / storm magnitude across matched events
- `noaa_log_damage`: log(1 + NOAA damage estimate) — independent severity signal
- `noaa_log_deaths`: log(1 + total deaths) — human toll proxy
- `noaa_log_injuries`: log(1 + total injuries) — severity proxy
- `noaa_event_count`: number of distinct NOAA events matched — breadth of impact
- `noaa_matched`: binary flag — 1 if any NOAA event matched, 0 otherwise

## Label
Same 3-class funding tier as Phase 1 (Tier 3 excluded — Congressional supplemental):
- Tier 0: Minor    < $1M
- Tier 1: Moderate $1M–$50M
- Tier 2: Major    $50M–$500M

In [ ]:
import pandas as pd
import numpy as np
import os, sys, warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '../../')
from utils import DISASTER_BINS, DISASTER_LABELS, get_season, add_prior_disasters

PROCESSED = '../data/processed/'

CPI = {
    1998:163.0,1999:166.6,2000:172.2,2001:177.1,2002:179.9,2003:184.0,
    2004:188.9,2005:195.3,2006:201.6,2007:207.3,2008:215.3,2009:214.5,
    2010:218.1,2011:224.9,2012:229.6,2013:233.0,2014:236.7,2015:237.0,
    2016:240.0,2017:245.1,2018:251.1,2019:255.7,2020:258.8,2021:270.97,
    2022:292.66,2023:304.70
}
CPI_2019 = 255.7

## Load Merged Dataset

In [ ]:
df = pd.read_csv(PROCESSED + 'fema_noaa_merged.csv', low_memory=False)
print(f'Loaded: {df.shape}')
print(f'NOAA match rate: {df["noaa_matched"].mean():.1%}')

## Hydrometeorological Filter

Restrict to event types covered by NOAA Storm Events.
Earthquake, Biological, and other non-hydrometeorological types are excluded
because their NOAA features would be structurally zero — not reflecting low
intensity but simply missing data coverage.

In [ ]:
# Hydrometeorological event types covered by NOAA Storm Events
# These map to Phase 1's canonical incident type categories
HYDRO_TYPES = [
    'Hurricane',
    'Flood',
    'Severe Storm',
    'Winter Storm',
    'Wildfire',
    'Tornado',
    'Coastal Storm',
    'Typhoon',
    'Tropical Storm',
]

n_before = len(df)
df = df[df['incidentType'].isin(HYDRO_TYPES)].copy()
n_after  = len(df)

excluded = n_before - n_after
print(f'Before filter : {n_before:,} disasters')
print(f'After filter  : {n_after:,} disasters')
print(f'Excluded      : {excluded:,} ({excluded/n_before:.1%}) — non-hydrometeorological events')
print(f'\nIncident type breakdown after filter:')
print(df['incidentType'].value_counts())
print(f'\nNote: Phase 2 covers ~{n_after/n_before:.0%} of original FEMA PA declarations')

## Clean Data
Parse dates, drop rows with missing funding or begin date, and fill NOAA nulls.
Numeric columns inherited from Phase 1 are filled with their median.

In [ ]:
# Parse dates
for col in ['incidentBeginDate','incidentEndDate','declarationDate']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Drop rows without dollar amount or begin date
df = df.dropna(subset=['total_federal_share', 'incidentBeginDate'])

# Fill NOAA nulls with 0 (unmatched = no recorded intensity)
NOAA_COLS = ['noaa_event_count','noaa_peak_magnitude','noaa_mean_magnitude',
             'noaa_total_damage_usd','noaa_max_deaths','noaa_total_deaths',
             'noaa_max_injuries']
for col in NOAA_COLS:
    if col in df.columns:
        df[col] = df[col].fillna(0)

df['noaa_matched'] = df['noaa_matched'].fillna(0).astype(int)

# Fill existing numeric nulls
NUM_COLS = ['population','median_income','poverty_rate','risk_score']
for col in NUM_COLS:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

print(f'After cleaning: {df.shape}')

## Feature Engineering
Engineer time-based features, prior disaster counts, CPI-adjusted dollar amounts,
and log-transformed NOAA columns to handle right-skewed damage/death distributions.

In [ ]:
# Time features
df['incident_year']          = df['incidentBeginDate'].dt.year
df['incident_season']        = df['incidentBeginDate'].apply(get_season)
df['declaration_lag_days']   = (df['declarationDate'] - df['incidentBeginDate']).dt.days.clip(lower=0)
df['incident_duration_days'] = (df['incidentEndDate'] - df['incidentBeginDate']).dt.days.clip(lower=0)

# Prior disasters
df = add_prior_disasters(df, state_col='stateAbbreviation',
                         date_col='incidentBeginDate', window_years=5)

# CPI adjustment
df['cpi_year'] = df['incident_year'].map(CPI).fillna(CPI_2019)
df['total_federal_share_2019'] = df['total_federal_share'] * (CPI_2019 / df['cpi_year'])

# NOAA log-transform (right-skewed)
df['noaa_log_damage']   = np.log1p(df['noaa_total_damage_usd'])
df['noaa_log_deaths']   = np.log1p(df['noaa_total_deaths'])
df['noaa_log_injuries'] = np.log1p(df['noaa_max_injuries'])

# Funding tier label (3-class, Tier 3 excluded)
BINS   = [0, 1_000_000, 50_000_000, 500_000_000, float('inf')]
LABELS = [0, 1, 2, 3]
df['funding_tier'] = pd.cut(df['total_federal_share_2019'],
                             bins=BINS, labels=LABELS).astype('Int64')
df = df[df['funding_tier'] <= 2].copy()
df['funding_tier'] = df['funding_tier'].astype(int)

print(f'Final shape: {df.shape}')
print(df['funding_tier'].value_counts().sort_index())

## Save Cleaned Dataset

In [ ]:
df.to_csv(PROCESSED + 'cleaned_fema_noaa.csv', index=False)
print(f'Saved: cleaned_fema_noaa.csv ({len(df):,} rows)')